# 1-WL Structural Signatures vs. Graph Convolutional Networks for Fraud Detection

**Research question.** Does 1-Weisfeiler-Leman neighbourhood structure add predictive signal
beyond node-level transaction features, and how does it compare with a GCN baseline?

**Datasets.** Elliptic Bitcoin (fraud detection, temporal split) and Cora (positive control).

**Structure**
1. Setup and shared abstractions
2. Data loading
3. Structural characterisation of the graphs
4. 1-WL: discrete refinement and its continuous relaxation
5. GCN baseline
6. Elliptic: feature ablation, quantisation sweep, aggregation ceiling, significance
7. Cora: positive control
8. Computational cost
9. Summary

## 1. Setup

In [ ]:
import time
import warnings
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.sparse import coo_matrix, csr_matrix
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import OneHotEncoder
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Both datasets are wrapped in a single immutable container and every routine below takes it as an
explicit argument. Nothing depends on notebook-level state, so the two graphs cannot be confused
with one another.

In [ ]:
@dataclass(frozen=True)
class GraphData:
    name: str
    X: np.ndarray          # node features
    y: np.ndarray          # labels; -1 = unlabelled
    src: np.ndarray        # undirected edge endpoints
    dst: np.ndarray
    A: csr_matrix          # symmetric adjacency
    deg: np.ndarray
    train: np.ndarray      # full training mask (model fitting)
    train_inner: np.ndarray  # subset of train, used to select GCN epoch count
    val: np.ndarray        # validation mask (epoch selection only)
    test: np.ndarray
    binary: bool


def build_graph(name, X, y, edge_index, train, train_inner, val, test, binary):
    n = X.shape[0]
    ei = to_undirected(torch.as_tensor(edge_index, dtype=torch.long), num_nodes=n).numpy()
    src, dst = ei[0], ei[1]
    A = coo_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(n, n)).tocsr()
    return GraphData(name, X.astype(np.float32), y, src, dst, A,
                     np.asarray(A.sum(1)).ravel(), train, train_inner, val, test, binary)

## 2. Data

**Elliptic.** 203,769 Bitcoin transactions over 49 time steps. Features are split into 93 local
transaction attributes and 72 aggregated neighbourhood statistics supplied by the dataset authors.
the two are kept separate so the graph's contribution can be isolated. Following ***Weber et al. (2019)***,
time steps 1-34 are used for training, 30-34 are used for validation and 35-49 for testing. Unlabelled nodes remain in the graph they carry messages and colours but are never trained on or scored.

In [ ]:
def load_elliptic(path="../data"):
    feats = pd.read_csv(f"{path}/elliptic_txs_features.csv", header=None)
    edges = pd.read_csv(f"{path}/elliptic_txs_edgelist.csv")
    classes = pd.read_csv(f"{path}/elliptic_txs_classes.csv")

    index = {tx: i for i, tx in enumerate(feats[0].values)}
    n = len(index)

    classes["node_idx"] = classes["txId"].map(index)
    classes = classes.sort_values("node_idx")
    assert (classes["node_idx"].values == np.arange(n)).all(), "class file does not cover every node"
    y = classes["class"].astype(str).map({"1": 1, "2": 0, "unknown": -1}).values.astype(np.int64)

    X = feats.iloc[:, 2:].values.astype(np.float32)
    step = feats.iloc[:, 1].values

    e = edges.copy()
    e["txId1"] = e["txId1"].map(index)
    e["txId2"] = e["txId2"].map(index)
    e = e.dropna().astype(np.int64)

    labelled = y != -1
    g = build_graph(
        name="Elliptic", X=X, y=y, edge_index=e.values.T,
        train=labelled & (step <= 34),
        train_inner=labelled & (step <= 29),
        val=labelled & (step >= 30) & (step <= 34),
        test=labelled & (step >= 35),
        binary=True,
    )
    return g


elliptic = load_elliptic()
ELL_LOCAL = elliptic.X[:, :93]     # local transaction features
ELL_FULL = elliptic.X              # local + the authors' 72 aggregated features

print(f"Elliptic: {len(elliptic.y):,} nodes | {len(elliptic.src) // 2:,} edges | "
      f"{elliptic.X.shape[1]} features")
print(f"  illicit {int((elliptic.y == 1).sum()):,} | licit {int((elliptic.y == 0).sum()):,} | "
      f"unlabelled {int((elliptic.y == -1).sum()):,}")
print(f"  train {int(elliptic.train.sum()):,} (t<=34) | test {int(elliptic.test.sum()):,} (t>=35)")
assert not (elliptic.train & elliptic.test).any(), "train/test overlap"

**Cora.** A citation network used here purely as a positive control. It is strongly homophilous and its published results are well established (features-only ≈ $0.58$ accuracy, GCN ≈ $0.81$, ***Kipf & Welling, 2017***), so it establishes whether the pipeline extracts graph signal when graph signal is known to exist.

In [ ]:
def load_cora():
    d = Planetoid(root="data/Planetoid", name="Cora")[0]
    m = lambda t: t.numpy()
    return build_graph(
        name="Cora", X=d.x.numpy(), y=d.y.numpy(), edge_index=d.edge_index.numpy(),
        train=m(d.train_mask), train_inner=m(d.train_mask),
        val=m(d.val_mask), test=m(d.test_mask), binary=False,
    )


cora = load_cora()

print(f"Cora: {len(cora.y):,} nodes | {len(cora.src) // 2:,} edges | "
      f"{cora.X.shape[1]} features | {len(set(cora.y))} classes")
print(f"  train {int(cora.train.sum())} | test {int(cora.test.sum())}")

## 3. Structural characterisation

Raw edge homophily is not comparable across datasets: Elliptic is 90% licit, so two random labelled
nodes already share a class 82% of the time, whereas Cora's seven balanced classes give a chance
baseline near 18%. Homophily is therefore reported adjusted for class prior,
$h_{adj} = (h - h_{chance}) / (1 - h_{chance})$ with $h_{chance} = \sum_c p_c^2$.

In [ ]:
def structural_profile(g):
    labelled = g.y != -1
    lab_neigh = np.asarray(g.A @ labelled.astype(np.float32)).ravel()

    both = labelled[g.src] & labelled[g.dst]
    h_obs = float(((g.y[g.src] == g.y[g.dst]) & both).sum() / both.sum())
    _, counts = np.unique(g.y[labelled], return_counts=True)
    h_chance = float(((counts / counts.sum()) ** 2).sum())

    return {
        "dataset": g.name,
        "nodes": f"{len(g.y):,}",
        "edges": f"{len(g.src) // 2:,}",
        "mean degree": round(float(g.deg.mean()), 2),
        "degree <= 2": f"{(g.deg <= 2).mean():.1%}",
        "homophily (raw)": round(h_obs, 3),
        "homophily (adjusted)": round((h_obs - h_chance) / (1 - h_chance), 3),
        "labelled-neighbour coverage": f"{(lab_neigh[labelled] >= 1).mean():.1%}",
    }


print("STRUCTURAL PROFILE\n")
print(pd.DataFrame([structural_profile(elliptic), structural_profile(cora)])
      .set_index("dataset").T.to_string())

## 4. 1-WL signatures

The 1-WL refinement rule maps each node to a hash of *(own colour, multiset of neighbour colours)*.
This project compares two distinct representations of that rule.

**Discrete (`wl_colours`).** The multiset is hashed to a single integer, exactly as in the classical
algorithm. Because Elliptic's features are continuous, they must first be quantised into `K` discrete
colours by k-means, a step 1-WL requires and which is examined in **6.2**.

**Relaxed (`wl_histogram`).** The multiset is retained as a `K`-dimensional count vector rather than
hashed. This is the aggregation a message-passing network performs ***(Xu et al., 2019)*** and it avoids
the colour explosion the discrete form suffers.

Colour vocabularies are built from features and edges only, no labels are involved at any stage. They
are fitted over all nodes, matching the transductive setting in which the GCN operates.

In [ ]:
def wl_colours(g, X_seed, K, rounds, seed=SEED):
    km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=10, batch_size=4096)
    colours = km.fit_predict(X_seed).astype(np.int64)
    n = len(colours)
    neighbours = [g.A.indices[g.A.indptr[v]:g.A.indptr[v + 1]] for v in range(n)]

    history = [colours.copy()]
    for _ in range(rounds):
        table, refined = {}, np.empty(n, dtype=np.int64)
        for v in range(n):
            signature = (colours[v], tuple(sorted(colours[neighbours[v]])))
            if signature not in table:
                table[signature] = len(table)
            refined[v] = table[signature]
        colours = refined
        history.append(colours.copy())
    return history


def wl_onehot(history, fit_mask, min_count=10):
    blocks = []
    for colours in history:
        kept = {c for c, k in Counter(colours[fit_mask]).items() if k >= min_count}
        folded = np.array([c if c in kept else -1 for c in colours]).reshape(-1, 1)
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        encoder.fit(folded[fit_mask])
        blocks.append(encoder.transform(folded).astype(np.float32))
    return np.hstack(blocks)


def wl_histogram(g, X_seed, K, rounds=2, seed=SEED):
    km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=10, batch_size=4096)
    colours = km.fit_predict(X_seed).astype(np.int64)
    n = len(colours)

    onehot = np.zeros((n, K), dtype=np.float32)
    onehot[np.arange(n), colours] = 1.0
    degree = np.maximum(g.deg, 1.0)[:, None]

    blocks, current = [onehot], onehot
    for _ in range(rounds):
        current = np.asarray(g.A @ current, dtype=np.float32)
        blocks += [current, current / degree]          # counts and proportions
    return np.hstack(blocks).astype(np.float32)


def neighbour_means(g, X_seed, rounds=2):
    """Aggregation with no quantisation: the K -> infinity limit of wl_histogram."""
    degree = np.maximum(g.deg, 1.0)[:, None]
    blocks, current = [], X_seed
    for _ in range(rounds):
        current = np.asarray(g.A @ current, dtype=np.float32) / degree
        blocks.append(current)
    return np.hstack(blocks).astype(np.float32)

## 5. Evaluation and models

In [ ]:
def evaluate(g, X, seeds=3):
    primary, secondary = [], []
    for s in range(seeds):
        rf = RandomForestClassifier(n_estimators=100, random_state=s, n_jobs=-1,
                                    class_weight="balanced")
        rf.fit(X[g.train], g.y[g.train])
        pred = rf.predict(X[g.test])
        if g.binary:
            primary.append(average_precision_score(
                g.y[g.test], rf.predict_proba(X[g.test])[:, 1]))
            secondary.append(f1_score(g.y[g.test], pred, pos_label=1, zero_division=0))
        else:
            primary.append(accuracy_score(g.y[g.test], pred))
            secondary.append(f1_score(g.y[g.test], pred, average="macro"))

    key = "AUPRC" if g.binary else "accuracy"
    second = "illicit F1" if g.binary else "macro F1"
    return {"dims": X.shape[1],
            key: f"{np.mean(primary):.4f} ± {np.std(primary):.4f}",
            second: f"{np.mean(secondary):.4f}"}


def test_scores(g, X, seeds=5):
    runs = []
    for s in range(seeds):
        rf = RandomForestClassifier(n_estimators=100, random_state=s, n_jobs=-1,
                                    class_weight="balanced")
        rf.fit(X[g.train], g.y[g.train])
        runs.append(rf.predict_proba(X[g.test])[:, 1])
    return np.mean(runs, axis=0)


def paired_bootstrap(scores_a, scores_b, y, n_boot=1000, seed=SEED):
    """95% CI for AUPRC(a) - AUPRC(b), resampling test nodes."""
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() == 0:
            continue
        diffs.append(average_precision_score(y[i], scores_a[i])
                     - average_precision_score(y[i], scores_b[i]))
    diffs = np.asarray(diffs)
    return diffs.mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

The GCN baseline is a two-layer graph convolution with inverse-frequency class weights. The number of
epochs is selected on a held-out validation window (Elliptic: t = 30–34) and the model is then
retrained from scratch on the full training set for that many epochs, so it sees exactly the same
training data as the Random Forest. The test set is never used for model selection.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_dim, n_classes, hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, n_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)


def train_gcn(g, X, max_epochs=300, lr=0.01, weight_decay=5e-4):
    x = torch.tensor(X)
    edge_index = torch.tensor(np.stack([g.src, g.dst]), dtype=torch.long)
    y = torch.tensor(np.where(g.y < 0, 0, g.y), dtype=torch.long)
    n_classes = int(y[g.train].max()) + 1

    counts = torch.bincount(y[torch.tensor(g.train_inner)], minlength=n_classes).float()
    weight = counts.sum() / (n_classes * counts)

    def fit(mask, epochs, select=False):
        torch.manual_seed(SEED)
        model = GCN(X.shape[1], n_classes)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        m = torch.tensor(mask)
        best_score, best_epoch = -1.0, epochs

        for epoch in range(1, epochs + 1):
            model.train()
            opt.zero_grad()
            F.cross_entropy(model(x, edge_index)[m], y[m], weight=weight).backward()
            opt.step()

            if select:
                model.eval()
                with torch.no_grad():
                    prob = F.softmax(model(x, edge_index), dim=1).numpy()
                score = (average_precision_score(g.y[g.val], prob[g.val, 1]) if g.binary
                         else accuracy_score(g.y[g.val], prob[g.val].argmax(1)))
                if score > best_score:
                    best_score, best_epoch = score, epoch
        return model, best_epoch

    _, best_epoch = fit(g.train_inner, max_epochs, select=True)
    model, _ = fit(g.train, best_epoch)

    model.eval()
    with torch.no_grad():
        logits = model(x, edge_index)
        prob = F.softmax(logits, dim=1).numpy()
        pred = logits.argmax(1).numpy()

    if g.binary:
        return {"epochs": best_epoch,
                "AUPRC": average_precision_score(g.y[g.test], prob[g.test, 1]),
                "illicit F1": f1_score(g.y[g.test], pred[g.test], pos_label=1, zero_division=0),
                "precision": precision_score(g.y[g.test], pred[g.test], pos_label=1, zero_division=0),
                "recall": recall_score(g.y[g.test], pred[g.test], pos_label=1, zero_division=0)}
    return {"epochs": best_epoch,
            "accuracy": accuracy_score(g.y[g.test], pred[g.test]),
            "macro F1": f1_score(g.y[g.test], pred[g.test], average="macro")}

## 6. Elliptic

### 6.1 Discrete 1-WL does not survive refinement

Colours are counted as usable if they occur at least ten times in the training period, anything rarer
carries no learnable signal. The collapse below is symmetric across train and test, so it reflects
over-discrimination by the refinement rule rather than temporal drift.

In [ ]:
history = wl_colours(elliptic, ELL_LOCAL, K=32, rounds=3)

rows = []
for r, colours in enumerate(history):
    kept = {c for c, k in Counter(colours[elliptic.train]).items() if k >= 10}
    rows.append({
        "WL round": r,
        "distinct colours": f"{len(set(colours)):,}",
        "colours retained": len(kept),
        "train nodes covered": f"{np.mean([c in kept for c in colours[elliptic.train]]):.1%}",
        "test nodes covered": f"{np.mean([c in kept for c in colours[elliptic.test]]):.1%}",
    })

print("COLOUR REFINEMENT UNDER DISCRETE 1-WL (Elliptic, K=32)\n")
print(pd.DataFrame(rows).to_string(index=False))

### 6.2 Feature ablation

`degree only` is a control, a uniform initial colouring reduces 1-WL to node degree, so any improvement it produces is not attributable to the features.

`raw neighbour means` removes the quantisation entirely and is therefore an upper bound on what any neighbourhood aggregation over this graph can achieve.

In [ ]:
wl_disc = wl_onehot(history, elliptic.train)
wl_hist = wl_histogram(elliptic, ELL_LOCAL, K=64)
wl_deg = wl_onehot([elliptic.deg.astype(np.int64)], elliptic.train)
nbr_mean = neighbour_means(elliptic, ELL_LOCAL)

conditions = {
    "Local features only":                 ELL_LOCAL,
    "Local + degree (control)":            np.hstack([ELL_LOCAL, wl_deg]),
    "Local + 1-WL discrete":               np.hstack([ELL_LOCAL, wl_disc]),
    "Local + 1-WL histogram (K=64)":       np.hstack([ELL_LOCAL, wl_hist]),
    "Local + raw neighbour means":         np.hstack([ELL_LOCAL, nbr_mean]),
    "Local + aggregated (Weber et al.)":   ELL_FULL,
    "Local + aggregated + 1-WL histogram": np.hstack([ELL_FULL, wl_hist]),
}

ablation = pd.DataFrame([{"condition": k, **evaluate(elliptic, v)}
                         for k, v in conditions.items()])

print("FEATURE ABLATION (Elliptic, Random Forest, temporal split)\n")
print(ablation.to_string(index=False))

### 6.3 Quantisation sweep

`K` controls how finely the continuous features are discretised before aggregation, and therefore how
much feature information 1-WL can carry. Its effect on Elliptic is compared with Cora in **7**.

In [ ]:
sweep_ell = []
for K in [16, 32, 64, 128, 256]:
    scores = [evaluate(elliptic, np.hstack([ELL_LOCAL, wl_histogram(elliptic, ELL_LOCAL, K, seed=s)]),
                       seeds=2)["AUPRC"] for s in range(2)]
    means = [float(s.split(" ± ")[0]) for s in scores]
    sweep_ell.append({"K": K, "AUPRC": f"{np.mean(means):.4f}"})

print("QUANTISATION SWEEP (Elliptic)\n")
print(pd.DataFrame(sweep_ell).to_string(index=False))
print("\nbaseline (no WL): 0.7816    aggregated features: 0.7934")

### 6.4 Statistical significance

Differences of this size cannot be interpreted without an interval. The test resamples test nodes
1,000 times and reports the 95% confidence interval on the paired difference in AUPRC.

In [ ]:
base = test_scores(elliptic, ELL_LOCAL)
wl = test_scores(elliptic, np.hstack([ELL_LOCAL, wl_hist]))
agg = test_scores(elliptic, ELL_FULL)
y_test = elliptic.y[elliptic.test]

comparisons = [
    ("1-WL histogram  vs  local only", wl, base),
    ("Aggregated      vs  local only", agg, base),
    ("1-WL histogram  vs  aggregated", wl, agg),
]

rows = []
for label, a, b in comparisons:
    delta, lo, hi = paired_bootstrap(a, b, y_test)
    rows.append({"comparison": label, "Δ AUPRC": f"{delta:+.4f}",
                 "95% CI": f"[{lo:+.4f}, {hi:+.4f}]",
                 "significant": "yes" if (lo > 0 or hi < 0) else "no"})

print("PAIRED BOOTSTRAP, 1,000 RESAMPLES (Elliptic)\n")
print(pd.DataFrame(rows).to_string(index=False))

### 6.5 GCN baseline

In [ ]:
gcn_ell = train_gcn(elliptic, ELL_FULL)

print("GCN BASELINE (Elliptic)\n")
print(f"  epochs selected on validation : {gcn_ell['epochs']}")
print(f"  AUPRC                         : {gcn_ell['AUPRC']:.4f}")
print(f"  illicit F1                    : {gcn_ell['illicit F1']:.4f}")
print(f"  precision                     : {gcn_ell['precision']:.4f}")
print(f"  recall                        : {gcn_ell['recall']:.4f}")

Reference ***(Weber et al., 2019, Table 1)***: GCN illicit F1 $0.628$ (precision $0.812$, recall $0.512$); Skip-GCN $0.705$; Random Forest $0.788$.
This GCN reproduces the published figure, the precision/recall balance differs because the loss is weighted by inverse class frequency rather than the authors' $0.3/0.7$ ratio.

## 7. Cora: positive control

Cora establishes that the pipeline recovers graph signal where graph signal is known to be present.
The published references are = $0.58$ accuracy for a features-only classifier and = $0.81$ for a GCN.

In [ ]:
cora_hist = wl_histogram(cora, cora.X, K=256)

cora_results = pd.DataFrame([
    {"condition": "Features only", **evaluate(cora, cora.X)},
    {"condition": "Features + 1-WL histogram (K=256)", **evaluate(cora, np.hstack([cora.X, cora_hist]))},
])

gcn_cora = train_gcn(cora, cora.X)

print("POSITIVE CONTROL (Cora)\n")
print(cora_results.to_string(index=False))
print(f"\n  GCN: accuracy {gcn_cora['accuracy']:.4f} | macro F1 {gcn_cora['macro F1']:.4f} "
      f"({gcn_cora['epochs']} epochs)")
cora_baseline = float(evaluate(cora, cora.X)["accuracy"].split(" ± ")[0])
print(f"\\nbaseline (no WL): {cora_baseline:.4f}    GCN: {gcn_cora['accuracy']:.4f}")

In [ ]:
sweep_cora = []
for K in [8, 16, 32, 64, 128, 256, 512]:
    accs = []
    for s in range(3):
        X = np.hstack([cora.X, wl_histogram(cora, cora.X, K, seed=s)])
        accs.append(float(evaluate(cora, X, seeds=2)["accuracy"].split(" ± ")[0]))
    sweep_cora.append({"K": K, "accuracy": f"{np.mean(accs):.4f} ± {np.std(accs):.4f}"})

print("QUANTISATION SWEEP (Cora)\n")
print(pd.DataFrame(sweep_cora).to_string(index=False))
print(f"\nbaseline (no WL): 0.5762    GCN: {gcn_cora['accuracy']:.4f}")

## 8. Computational cost

The 1-WL pipeline is non-parametric and its feature construction is a one-off preprocessing cost,
which is reported separately rather than hidden.

In [ ]:
X_wl = np.hstack([ELL_FULL, wl_hist])

t0 = time.time()
wl_histogram(elliptic, ELL_LOCAL, K=64)
wl_build = time.time() - t0

rf_train, rf_infer = [], []
for s in range(5):
    rf = RandomForestClassifier(n_estimators=100, random_state=s, n_jobs=-1,
                                class_weight="balanced")
    t0 = time.time(); rf.fit(X_wl[elliptic.train], elliptic.y[elliptic.train])
    rf_train.append(time.time() - t0)
    t0 = time.time(); rf.predict(X_wl[elliptic.test])
    rf_infer.append(time.time() - t0)

x = torch.tensor(ELL_FULL)
ei = torch.tensor(np.stack([elliptic.src, elliptic.dst]), dtype=torch.long)
y_t = torch.tensor(np.where(elliptic.y < 0, 0, elliptic.y), dtype=torch.long)
tm = torch.tensor(elliptic.train)

gcn_train, gcn_infer = [], []
for s in range(5):
    torch.manual_seed(s)
    model = GCN(ELL_FULL.shape[1], 2)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    t0 = time.time()
    model.train()
    for _ in range(50):
        opt.zero_grad()
        F.cross_entropy(model(x, ei)[tm], y_t[tm]).backward()
        opt.step()
    gcn_train.append(time.time() - t0)

    model.eval()
    t0 = time.time()
    with torch.no_grad():
        model(x, ei)
    gcn_infer.append(time.time() - t0)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("COMPUTATIONAL COST (CPU)\n")
print(pd.DataFrame([
    {"metric": "trainable parameters", "GCN": f"{n_params:,}", "1-WL + Random Forest": "non-parametric"},
    {"metric": "feature construction", "GCN": "—", "1-WL + Random Forest": f"{wl_build:.2f}s"},
    {"metric": "training (50 epochs)", "GCN": f"{np.mean(gcn_train):.2f}s ± {np.std(gcn_train):.2f}",
     "1-WL + Random Forest": f"{np.mean(rf_train):.2f}s ± {np.std(rf_train):.2f}"},
    {"metric": "inference", "GCN": f"{np.mean(gcn_infer):.3f}s ± {np.std(gcn_infer):.3f}",
     "1-WL + Random Forest": f"{np.mean(rf_infer):.3f}s ± {np.std(rf_infer):.3f}"},
]).to_string(index=False))

## 9. Summary
 
|                                   | Elliptic          | Cora             |
|-----------------------------------|-------------------|------------------|
| mean degree                       | 2.30              | 3.90             |
| nodes with degree <= 2            | 79.0%             | 39.4%            |
| adjusted homophily                | 0.737             | 0.768            |
| labelled-neighbour coverage       | 77.0%             | 100%             |
| baseline without graph            | 0.7816 (AUPRC)    | 0.5643 (accuracy)|
| best 1-WL histogram               | 0.7875            | 0.7182           |
| GCN                               | 0.6299            | 0.8090           |
| range across the quantisation sweep | 0.002           | 0.151            |
 
### Validation
 
The GCN reaches illicit F1 $0.6208$ against the $0.628$ reported by ***Weber et al. (2019)***, and the Random
Forest on all features reaches $0.8124$ against their $0.788$. On Cora the GCN reaches $0.8090$ accuracy
against the $~0.81$ of ***Kipf and Welling (2017)***. All three published results are reproduced, so the
pipeline is sound and the findings below are properties of the data rather than of the implementation.
 
### Findings
 
1. **Discrete 1-WL over-discriminates.** By the second refinement round $134,939$ distinct colours exist
   across $203,769$ nodes and only $96$ occur often enough to be usable, covering $7.7%$ of training nodes.
   The collapse is symmetric across train and test, so it reflects the refinement rule rather than
   temporal drift. As a feature set it performs slightly below the baseline ($0.7807 vs 0.7816$).
 
2. **The relaxed formulation gives a small but significant improvement.** Retaining the neighbour
   multiset as a histogram rather than hashing it yields Δ AUPRC $+0.0059 (95% CI [+0.0035, +0.0081]$).
   This recovers roughly half of the $+0.0117$ that the authors' 72 hand-engineered aggregated features
   provide, using two hyperparameters and no domain knowledge. It does not match them
   ($Δ −0.0058, CI [−0.0095, −0.0023]$) and adds nothing once they are present ($0.7913 vs 0.7934$).
 
3. **Quantisation binds on Cora but not on Elliptic.** The parameter K moves accuracy by 15 points on
   Cora ($0.5677 → 0.7182$) and AUPRC by $0.002$ on Elliptic. Removing the quantisation entirely (raw
   neighbour means) reaches only $0.7885$ on Elliptic, confirming that discretisation is not the binding
   constraint there. The slope of the sweep therefore acts as a diagnostic for how much
   feature-correlated structure a graph carries.
 
4. **Elliptic is homophilous but its neighbourhoods are degenerate.** Once class imbalance is accounted
   for, its adjusted homophily ($0.737$) is comparable to Cora's ($0.768$)  the raw figures ($0.954 vs
   0.810$) invert this and are misleading. What distinguishes the two graphs is that $79%$ of Elliptic's
   nodes have at most two neighbours, and that its local features already reach $0.7816$ AUPRC unaided
   against Cora's $0.5643$. There is little headroom for the graph to fill.
 
5. **The aggregation function, not the graph, is the remaining lever.** Mean aggregation used by both
   the 1-WL histogram and the GCN reaches 0.7885. The authors' features, which aggregate the same
   neighbourhoods using maximum, minimum, standard deviation and correlation, reach $0.7934$. The graph
   is therefore not exhausted by mean-based methods, though the margin available is small.
 
### Conclusion
 
1-WL does not meaningfully improve fraud detection on Elliptic. The method is sound: on Cora it
recovers $63%$ of a $+0.245$ accuracy gain, and on Elliptic it recovers about half of the graph's total
measured value. That value is simply small bounded at roughly $+0.012$ AUPRC because the transaction
graph is sparse and the node features are already highly informative. ***Weber et al.*** reach the same
conclusion from the other direction: their Skip-GCN outperforms their GCN precisely because the skip
connection restores direct access to the node features.
 
### Limitations and future work
 
The Random Forest and the GCN differ in both classifier and representation, so only the
within-classifier comparisons isolate the contribution of the graph features. Colour vocabularies are
fitted over all nodes to match the GCN's transductive setting, a deployed system could not cluster on
future transactions, and a train-only variant would give a lower bound. The test window (t = 35–49)
contains the dark market shutdown at t = 43, which ***Weber et al.*** report degrades all methods and which
is not modelled here. Finally, the authors themselves describe their aggregated features as naive and
sub-optimal, yet no aggregation tested here surpasses them; since they differ chiefly in using several
aggregators rather than the mean alone, multiple-aggregator architectures (***Corso et al., 2020***) are the
natural next step.